# Python, part 4 · Patterns

**Goal:** read and write the patterns course 1 used to prove every ID is 8 digits and to recognize the date formats, and write the function that found those formats.

Patterns (*regular expressions*) look like line noise at first. You only need a handful of pieces to read nearly every pattern in course 1, and this notebook covers them.

**How to use this notebook** (the same as course 1)
- Start the lab cluster first (**Compute**; a cold start takes 5 minutes or more), then attach this notebook to it.
- Click a gray code cell and press **Shift + Enter** to run it. Go top to bottom.
- 🧪 **Your turn** cells contain a `___` blank. Replace it with your answer and run the cell; it tells you whether you're right. Answers are at the bottom if you're stuck.
- Nothing here can break anything. If things get strange, choose **Run → Clear state**, then run from the top.

> ⚠️ **Mock data only.** Everything in this course is fictional. Never load real offender data into any tool that isn't approved for criminal justice information.

In [ ]:
# ▶ SETUP: run this cell first (Shift + Enter). It finds the course files and builds today's data.
SESSION = "P4"

import sys
from pathlib import Path

def _find_course_files():
    for folder in [Path.cwd(), *Path.cwd().parents]:          # the course's Git folder in Databricks
        if (folder / "lnl" / "__init__.py").exists():
            return folder
    try:                                                         # found from the notebook's own location
        here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        for folder in Path("/Workspace" + here).parents:
            if (folder / "lnl" / "__init__.py").exists():
                return folder
    except Exception:
        pass
    raise RuntimeError("Can't find the course files. Open this notebook from inside the course's Git folder in Databricks.")

COURSE = _find_course_files()
sys.path.insert(0, str(COURSE))
from lnl import env
PATHS = env.prepare(SESSION)

In [ ]:
import re
import pandas as pd
from lnl import check, cleaning, dates, inside, pipeline
from lnl.check import ___

## 1. Remember Q4?
The Trust Test's Q4 asked: *what is the earliest projected release date?* Here it is asked of the dates as written, and after course 1's session 3:

In [ ]:
raw = pipeline.load_raw()
written = cleaning.strip_and_null(raw)["projected_release_date"]

print("Earliest, as written:     ", written.dropna().min())   # .dropna() leaves the empty values out
print("Earliest, after session 3:", pipeline.clean_through(3)["projected_release_date"].min())

As written, the "earliest" is `01-01-2018`, because text is sorted letter by letter, and `0` comes before `1` and `2`. To find the real earliest date, course 1 first had to work out how each date was written:

In [ ]:
written.dropna().sample(8, random_state=4).tolist()

There are 45,025 different dates written in many ways. The tool for "written like this" is a **pattern**.

## 2. Reading a pattern
A pattern describes what text looks like. These are the pieces course 1 uses:

| Piece | Means | Example | Matches |
| --- | --- | --- | --- |
| `\d` | any digit, 0 to 9 | `\d\d` | `07`, `42` |
| `{8}` | exactly 8 of the thing before it | `\d{8}` | `02381457` |
| `{1,2}` | one or two of it | `\d{1,2}` | `7`, `12` |
| `[A-Za-z]` | any one letter, capital or small | `[A-Za-z]{3}` | `Aug`, `AUG` |
| `+` | one or more of it | `[A-Za-z]+` | `February` |
| anything else | itself | `-`, `/`, `,`, a space | itself |

Write a pattern as a string with an `r` in front, `r"\d{8}"`, so Python leaves the backslashes alone.

`re.fullmatch(pattern, text)` asks whether the **whole** text fits the pattern. It gives back a match if it does, and `None` if it doesn't. `bool(...)` turns that into `True` or `False`:

In [ ]:
print(bool(re.fullmatch(r"\d{8}", "02381457")))
print(bool(re.fullmatch(r"\d{8}", "2381457")))      # only 7 digits
print(bool(re.fullmatch(r"\d{8}", "0238145O")))     # the last one is the letter O

## 3. Proving the IDs
Course 1 stored offender numbers as whole numbers, but only after **proving** that every one is exactly 8 digits. Load the file the naive way, the way session 1 did, and pandas guesses they're numbers. The leading zero is gone, and Q5's offender `02381457` can't be found:

In [ ]:
naive = pipeline.load_naive()
print(naive["offender_number"].head(3).tolist())

🧪 **Your turn #1.** Write the pattern for an ID: exactly 8 digits. The check tries it on every offender number and SID in the file, and on some near-misses (7 digits, 9 digits, a letter O, a trailing space...).

In [ ]:
answer_1 = ___      # a pattern string, starting r"
check.p4_ex1(answer_1)

Here is how course 1 uses its pattern, `ID_PATTERN = r"\d{8}"`, in `set_record_id`. Line 10 counts the IDs that are **not** (`~`) a full match, using `.str.fullmatch`, which is `re.fullmatch` for every value in a column:

In [ ]:
inside.show(cleaning.set_record_id)

If any ID failed, `raise` on line 15 would stop everything with an error rather than guess. In this file none fail: every one of the 100,000 offender numbers and 97,003 SIDs is 8 digits. Proof, not luck.

## 4. The shape of a date
Course 1 found the date formats with a trick: turn every digit into `9` and every letter into `A`. Then `03/14/2027` and `11/02/2031` both become `99/99/9999`, the same **shape**.

`re.sub(pattern, new, text)` replaces every part of the text that fits the pattern:

In [ ]:
print(re.sub(r"\d", "9", "03/14/2027"))
print(re.sub(r"\d", "9", "04-AUG-2030"))

🧪 **Your turn #2.** Finish `shape`. The digits are done. Make every letter, capital or small, an `A`. The check runs it on every date in the column and compares the result with course 1's `dates.profile_shapes`.

In [ ]:
def shape(text):
    """The shape of a value: every digit becomes 9 and every letter becomes A."""
    text = re.sub(r"\d", "9", text)
    text = ___      # every letter becomes A: re.sub with [A-Za-z]
    return text

check.p4_ex2(shape)

## 5. The payoff: every date, in a few shapes

In [ ]:
check.p4_payoff(shape)

45,025 different dates come down to 26 shapes, and those come down to nine formats. Here are the patterns course 1 wrote for them, one per format:

In [ ]:
for label, pattern, how_to_read in dates.DATE_FORMATS:
    print(f"{label:22} {pattern}")

You can read every one of them now. `DATE_FORMATS` is a list (part 2) of *tuples*: a tuple is a list in round brackets that can't be changed. Each holds a name, a pattern and the format code Python uses to read that kind of date.

🧪 **Your turn #3.** Write the pattern for a date written like `2029-11-03`. The check compares it with course 1's on all 45,025 different dates. Watch out: `2027-01-17 00:00:00` starts the same way, but it has a time on the end and must not match.

In [ ]:
answer_3 = ___      # four digits, a dash, two digits, a dash, two digits
check.p4_ex3(answer_3)

Here's how course 1 uses the list, in `classify_formats`. It loops over the formats (line 4) and labels each date with the first pattern it fully matches (lines 5 and 6):

In [ ]:
inside.show(dates.classify_formats)

Session 3 then read each date with its own format, and that's how it found Q4's real answer.

## Wrap-up
- A **pattern** describes what text looks like: `\d` is a digit, `{8}` means exactly 8, `[A-Za-z]` is a letter, `+` means one or more.
- `re.fullmatch` asks whether the whole text fits; `.str.fullmatch` asks it of every value in a column.
- `re.sub` replaces whatever fits. The **shape** trick (digits to 9, letters to A) reveals the formats in a messy column.
- Prove a rule before you rely on it: course 1 refused to convert an ID it couldn't prove.

**Next, part 5 · Your own session 2:** put your functions from parts 1 to 4 together into your own pipeline, check it against course 1's, and see the same steps in course 2's SQL.

## Answers

**#1**
```python
answer_1 = r"\d{8}"
```
**#2**
```python
text = re.sub(r"[A-Za-z]", "A", text)
```
**#3**
```python
answer_3 = r"\d{4}-\d{2}-\d{2}"
```